# Reading the Treasury Yield Curve Before Modeling

This notebook is a guided exploratory analysis of daily U.S. Treasury constant-maturity yields. The goal is not to build machine-learning features yet. The goal is to learn what the data looks like, what changes through time, and what financial structure is visible before we ask a model to forecast anything.

We use seven maturities: **3M, 6M, 1Y, 2Y, 5Y, 10Y, and 30Y**. Each row is one trading-day observation, and each value is an annualized yield in percent.

## Setup

The notebook uses the cleaned Phase 1 dataset and the reusable plotting code in `src/market_resonance/visualization/treasury_eda.py`. Running the cells below will recreate every final figure under `reports/figures/`.

In [ ]:
# ruff: noqa: E402, I001
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from IPython.display import Image, display  # noqa: E402
from market_resonance.data import MATURITIES  # noqa: E402
from market_resonance.visualization.treasury_eda import (  # noqa: E402
    add_curve_slope,
    daily_yield_changes,
    generate_treasury_eda_figures,
    load_treasury_dataset,
)

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "treasury_yields_daily.csv"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"

In [ ]:
yields = load_treasury_dataset(DATA_PATH)
yields.head()

In [ ]:
summary = {
    "rows": len(yields),
    "start_date": yields["date"].min().date(),
    "end_date": yields["date"].max().date(),
    "duplicate_dates": int(yields["date"].duplicated().sum()),
    "missing_values": int(yields.isna().sum().sum()),
}
summary

## 1. Historical yields: the level of rates changes by regime

The first view follows each maturity through time. Financially, this is a picture of borrowing costs across different horizons. Short maturities are closely tied to monetary policy. Long maturities also reflect inflation expectations, long-run growth expectations, and the compensation investors require for lending for many years.

Statistically, yield levels are highly persistent. They do not bounce randomly around one fixed average. Instead, they move through regimes: high-rate periods, low-rate periods, hiking cycles, cutting cycles, and crisis periods.

In [ ]:
generate_treasury_eda_figures(DATA_PATH, FIGURE_DIR)
display(Image(filename=FIGURE_DIR / "treasury_historical_yields.png"))

## 2. Example yield curves: one date, many maturities

A yield curve is a cross-section. Instead of asking how the 10-year yield changed through time, we ask what all maturities looked like on one date.

Financially, the shape matters. An upward-sloping curve usually means longer lending horizons earn higher yields. A flat curve means short and long yields are similar. An inverted curve means short yields are above longer yields, which often appears when markets expect future rates or growth to fall.

Statistically, this view reminds us that the seven columns are not unrelated variables. They are ordered points along a curve.

In [ ]:
display(Image(filename=FIGURE_DIR / "treasury_example_yield_curves.png"))

## 3. Correlation matrix: maturities move together

The correlation matrix measures how closely yield levels at different maturities move together. Values near 1 mean two maturities tend to be high and low at the same times.

Financially, high correlations make sense because all Treasury yields share big macro drivers: Federal Reserve policy, inflation expectations, growth expectations, and risk sentiment.

Statistically, high correlation means the columns contain overlapping information. That is not bad, but it matters for modeling because the model sees several related versions of the same broad interest-rate cycle.

In [ ]:
display(Image(filename=FIGURE_DIR / "treasury_maturity_correlation.png"))

## 4. Daily yield changes: most days are small, but tails matter

Yield levels tell us where rates are. Daily yield changes tell us how quickly rates move. We measure changes in **basis points**, where 1 basis point equals 0.01 percentage points.

Financially, this shows the typical daily shock size for each maturity. Most days are modest. But stressful periods can create unusually large moves.

Statistically, daily changes are closer to the kind of movement a forecasting model may need to understand later. The distributions also reveal outliers and fat tails: large moves happen more often than a simple bell curve would suggest.

In [ ]:
changes = daily_yield_changes(yields)
changes[["date", *MATURITIES]].head()

In [ ]:
display(Image(filename=FIGURE_DIR / "treasury_daily_change_distributions.png"))

## 5. Curve slope: 10Y minus 2Y

The 10Y-minus-2Y slope is a compact summary of the yield curve's shape. A positive value means the 10-year yield is above the 2-year yield. A negative value means the curve is inverted at those maturities.

Financially, this slope is watched because inversions have often appeared before slowdowns or recessions. It is not a clock or a guarantee; it is a market signal about expectations.

Statistically, the slope is a derived time series. It compresses two columns into one interpretable number, which is useful for exploration. In this phase, we are only inspecting it, not turning it into ML input features yet.

In [ ]:
with_slope = add_curve_slope(yields)
with_slope[["date", "2Y", "10Y", "10Y_minus_2Y"]].tail()

In [ ]:
display(Image(filename=FIGURE_DIR / "treasury_10y_minus_2y_slope.png"))

## What we learned before ML

- Treasury yields are ordered by maturity, so the columns form a curve rather than a random set of variables.
- Yield levels are persistent and regime-dependent.
- Maturities are strongly correlated, especially neighboring maturities.
- Daily yield changes are usually small but have meaningful outliers.
- The 10Y-minus-2Y slope gives a readable summary of curve steepness and inversion episodes.

That is enough for Phase 2. The next phase can start thinking about features, but this notebook deliberately stops before building them.